In [1]:
import pandas as pd
import numpy as np

In [2]:
apc = pd.read_csv('./data/mbta_apc_may_2025.csv', engine='pyarrow')

In [3]:
apc = apc.loc[apc['stop_seq_id'] != 9999]
apc

,trip_date,bus,route,variant,block,direction,trip,stop_id,stop_seq_id,stop_name,act_stop_time,act_dep_time,psgr_on,psgr_off,psgr_load,latitude,longitude
0,2025-05-01,1630,34,0,3203,2,435,10642,0,FOREST HILLS STATION UPPER B,2025-05-01 04:39:25,2025-05-01 04:39:25,0,0,0,42.29926,-71.11787
1,2025-05-01,1630,34,0,3203,2,435,596,1,3867 WASHINGTON ST OPP TOLLG,2025-05-01 04:40:48,2025-05-01 04:40:48,0,0,0,42.29553,-71.11853
2,2025-05-01,1630,34,0,3203,2,435,597,2,WASHINGTON ST @ LOCHDALE RD,2025-05-01 04:41:00,2025-05-01 04:41:00,0,0,0,42.29427,-71.11985
3,2025-05-01,1630,34,0,3203,2,435,598,3,WASHINGTON ST @ ARCHDALE RD,2025-05-01 04:41:15,2025-05-01 04:41:15,0,0,0,42.29295,-71.12125
4,2025-05-01,1630,34,0,3203,2,435,599,4,WASHINGTON ST @ MOSGROVE AVE,2025-05-01 04:41:29,2025-05-01 04:41:29,0,0,0,42.29170,-71.12258
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
792032,2025-05-31,1664,32,1,3059,1,2422,6468,10,HYDE PARK AVE @ WEST ST,2025-06-01 00:26:59,2025-06-01 00:27:08,1,0,2,42.26207,-71.12188
792033,2025-05-31,1664,32,1,3059,1,2422,6470,11,HYDE PARK AVE @ WESTMINSTER,2025-06-01 00:27:29,2025-06-01 00:27:29,0,0,2,42.26443,-71.12135
792034,2025-05-31,1664,32,1,3059,1,2422,6471,12,HYDE PARK AVE @ THATCHER ST,2025-06-01 00:27:38,2025-06-01 00:27:38,0,0,2,42.26534,-71.12115
792035,2025-05-31,1650,36,8,3029,1,2430,10820,0,VETERANS HOSPITAL WEST ROXBU,2025-06-01 00:24:21,2025-06-01 00:24:31,0,0,0,42.27522,-71.17211


In [4]:
# stops look up
stops = apc.groupby('stop_id').agg(
    stop_name = pd.NamedAgg('stop_name', 'first'),
    lat = pd.NamedAgg('latitude', 'mean'),
    lon = pd.NamedAgg('longitude', 'mean')
).reset_index()
stops.sort_values('stop_id')

,stop_id,stop_name,lat,lon
0,120,RIVERMOOR ST @ INDUSTRIAL PA,42.277725,-71.179394
1,129,RIVERMOOR ST @ CHARLES PARK,42.278360,-71.175376
2,137,CHARLES PARK RD @ RIVERMOOR,42.278783,-71.175482
3,185,MATTAPAN NORTH BUSWAY,42.267711,-71.092449
4,334,ASHMONT BUSWAY,42.283992,-71.064008
...,...,...,...,...
780,96434,CUMMINS HWY @ RIVER ST,42.267838,-71.094782
781,96459,CUMMINS HWY @ AMERICAN LEGIO,42.278646,-71.116145
782,96500,AMERICAN LEGION HWY @ CANTER,42.285541,-71.110121
783,99832,245 WASHINGTON ST,42.251107,-71.169803


In [5]:
stops.to_csv('./data/stops.csv', index=False)

In [6]:
apc.columns

Index(['trip_date', 'bus', 'route', 'variant', 'block', 'direction', 'trip',
       'stop_id', 'stop_seq_id', 'stop_name', 'act_stop_time', 'act_dep_time',
       'psgr_on', 'psgr_off', 'psgr_load', 'latitude', 'longitude'],
      dtype='object')

In [7]:
apc.head(2)

,trip_date,bus,route,variant,block,direction,trip,stop_id,stop_seq_id,stop_name,act_stop_time,act_dep_time,psgr_on,psgr_off,psgr_load,latitude,longitude
0,2025-05-01,1630,34,0,3203,2,435,10642,0,FOREST HILLS STATION UPPER B,2025-05-01 04:39:25,2025-05-01 04:39:25,0,0,0,42.29926,-71.11787
1,2025-05-01,1630,34,0,3203,2,435,596,1,3867 WASHINGTON ST OPP TOLLG,2025-05-01 04:40:48,2025-05-01 04:40:48,0,0,0,42.29553,-71.11853


In [8]:
apc.dtypes

trip_date               object
bus                      int64
route                    int64
variant                  int64
block                    int64
direction                int64
trip                     int64
stop_id                  int64
stop_seq_id              int64
stop_name               object
act_stop_time    datetime64[s]
act_dep_time     datetime64[s]
psgr_on                  int64
psgr_off                 int64
psgr_load                int64
latitude               float64
longitude              float64
dtype: object

In [9]:
# trips look up
trips = apc.copy()
trip_id_cols = ['trip_date', 'bus', 'route', 'variant', 'block', 'direction', 'trip']
trips = trips.groupby(trip_id_cols).agg(
    trip_start_time_actual = pd.NamedAgg('act_dep_time', 'min'),
    trip_end_time_actual = pd.NamedAgg('act_stop_time', 'max'),
    total_boardings = pd.NamedAgg('psgr_on', 'sum'),
    num_stops = pd.NamedAgg('stop_id', 'count')
).reset_index()
trips['runtime_min'] = ((trips.trip_end_time_actual - trips.trip_start_time_actual).dt.seconds / 60).astype(int)
trips['trip_start_hour'] = trips.trip_start_time_actual.dt.hour
trips

,trip_date,bus,route,variant,block,direction,trip,trip_start_time_actual,trip_end_time_actual,total_boardings,num_stops,runtime_min,trip_start_hour
0,2025-05-01,1600,24,2,3028,1,535,2025-05-01 05:34:33,2025-05-01 05:59:45,31,38,25,5
1,2025-05-01,1600,24,2,3028,1,640,2025-05-01 06:39:27,2025-05-01 07:12:31,58,38,33,6
2,2025-05-01,1600,24,2,3028,1,810,2025-05-01 08:16:01,2025-05-01 08:56:49,31,36,40,8
3,2025-05-01,1600,24,2,3028,1,935,2025-05-01 09:39:33,2025-05-01 10:10:41,21,38,31,9
4,2025-05-01,1600,24,2,3028,1,1045,2025-05-01 10:51:51,2025-05-01 11:20:51,21,38,29,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...
25411,2025-05-31,1716,37,0,3129,2,1915,2025-05-31 19:13:58,2025-05-31 19:32:37,16,29,18,19
25412,2025-05-31,1716,51,0,3129,1,2040,2025-05-31 20:43:44,2025-05-31 21:13:37,7,47,29,20
25413,2025-05-31,1716,51,0,3129,1,2145,2025-05-31 21:53:00,2025-05-31 22:24:33,11,47,31,21
25414,2025-05-31,1716,51,0,3129,2,2010,2025-05-31 20:13:01,2025-05-31 20:43:14,7,46,30,20


In [10]:
trips.loc[trips['runtime_min'] < 1]

,trip_date,bus,route,variant,block,direction,trip,trip_start_time_actual,trip_end_time_actual,total_boardings,num_stops,runtime_min,trip_start_hour
1030,2025-05-01,1715,24,2,3026,1,1655,2025-05-01 17:19:03,2025-05-01 17:19:05,0,2,0,17
1040,2025-05-01,1715,24,2,3026,2,1605,2025-05-01 16:30:12,2025-05-01 16:30:13,18,2,0,16
1087,2025-05-02,1216,32,1,3077,2,1741,2025-05-02 17:17:13,2025-05-02 17:17:13,0,1,0,17
1154,2025-05-02,1604,40,0,3157,1,1306,2025-05-02 13:27:24,2025-05-02 13:27:26,0,2,0,13
1155,2025-05-02,1604,40,0,3157,1,1406,2025-05-02 14:28:34,2025-05-02 14:28:34,0,1,0,14
...,...,...,...,...,...,...,...,...,...,...,...,...,...
24506,2025-05-30,1692,24,2,3027,2,1805,2025-05-30 18:25:06,2025-05-30 18:25:09,0,2,0,18
24636,2025-05-30,1717,32,1,3097,1,1343,2025-05-30 14:19:56,2025-05-30 14:19:58,0,2,0,14
25030,2025-05-31,1658,30,2,3027,1,1730,2025-05-31 17:38:17,2025-05-31 17:38:17,0,1,0,17
25129,2025-05-31,1672,30,2,3027,1,1730,2025-05-31 17:48:27,2025-05-31 17:48:30,0,2,0,17


In [11]:
trips.loc[trips['runtime_min'] > 120].runtime_min.describe()

count     266.000000
mean     1432.011278
std        10.135491
min      1377.000000
25%      1428.000000
50%      1437.500000
75%      1439.000000
max      1439.000000
Name: runtime_min, dtype: float64

In [12]:
trips = trips.loc[(trips['runtime_min'] > 1) & (trips['runtime_min'] < 120)]
trips['trip_unique_id'] = trips['trip_date'].astype(str) + '_' + trips['bus'].astype(str) + '_' + trips['route'].astype(str) + '_' + trips['variant'].astype(str) + '_' + trips['block'].astype(str) + '_' + trips['direction'].astype(str) + '_' + trips['trip'].astype(str)
trips.to_csv('./data/trips.csv', index=False)

/tmp/ipykernel_2492/632351619.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  trips['trip_unique_id'] = trips['trip_date'].astype(str) + '_' + trips['bus'].astype(str) + '_' + trips['route'].astype(str) + '_' + trips['variant'].astype(str) + '_' + trips['block'].astype(str) + '_' + trips['direction'].astype(str) + '_' + trips['trip'].astype(str)


In [13]:
# filter and sort
sort_cols = trip_id_cols + ['stop_seq_id']
cols_to_keep = sort_cols + ['stop_id', 'act_stop_time', 'act_dep_time', 'psgr_on', 'psgr_off', 'psgr_load']
apc['trip_unique_id'] = apc['trip_date'].astype(str) + '_' + apc['bus'].astype(str) + '_' + apc['route'].astype(str) + '_' + apc['variant'].astype(str) + '_' + apc['block'].astype(str) + '_' + apc['direction'].astype(str) + '_' + apc['trip'].astype(str)
apc = apc.loc[apc['trip_unique_id'].isin(trips['trip_unique_id'])]
apc = apc[cols_to_keep].sort_values(sort_cols)
apc.to_csv('./data/apc.csv', index=False)

In [14]:
apc

,trip_date,bus,route,variant,block,direction,trip,stop_seq_id,stop_id,act_stop_time,act_dep_time,psgr_on,psgr_off,psgr_load
1022,2025-05-01,1600,24,2,3028,1,535,0,6411,2025-05-01 05:31:25,2025-05-01 05:34:33,2,0,2
1023,2025-05-01,1600,24,2,3028,1,535,1,6413,2025-05-01 05:35:29,2025-05-01 05:35:29,0,0,2
1024,2025-05-01,1600,24,2,3028,1,535,2,6415,2025-05-01 05:35:47,2025-05-01 05:35:47,0,0,2
1025,2025-05-01,1600,24,2,3028,1,535,3,6417,2025-05-01 05:36:07,2025-05-01 05:36:07,0,0,2
1026,2025-05-01,1600,24,2,3028,1,535,4,64171,2025-05-01 05:36:22,2025-05-01 05:36:22,0,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
790299,2025-05-31,1716,51,0,3129,2,2115,41,81916,2025-05-31 21:49:09,2025-05-31 21:49:09,0,0,2
790300,2025-05-31,1716,51,0,3129,2,2115,42,81917,2025-05-31 21:50:20,2025-05-31 21:50:20,0,0,2
790301,2025-05-31,1716,51,0,3129,2,2115,43,91916,2025-05-31 21:51:06,2025-05-31 21:51:06,0,0,2
790302,2025-05-31,1716,51,0,3129,2,2115,44,11917,2025-05-31 21:51:33,2025-05-31 21:51:33,0,0,2
